In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
# --------------------------------------------------
# 1. LOAD DATA
# --------------------------------------------------

df = pd.read_csv("customer_churn_day1.csv")

In [3]:
# --------------------------------------------------
# 2. UNDERSTAND DATA
# --------------------------------------------------

print(df.shape)
print(df.isnull().sum())
print(df["churn"].value_counts())

(505, 10)
customer_id          0
age                 10
tenure_months        0
monthly_charges      0
total_charges       12
contract_type        0
support_calls        0
internet_service    72
payment_method       6
churn                0
dtype: int64
churn
No     362
Yes    143
Name: count, dtype: int64


In [4]:
# --------------------------------------------------
# 3. CLEAN DATA
# --------------------------------------------------

df = df.drop_duplicates()

num_cols = df.select_dtypes(include="number").columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

df = df.dropna(subset=["churn"])

In [5]:
# --------------------------------------------------
# 4. ENCODE CATEGORICAL DATA
# --------------------------------------------------

df = df.drop("customer_id", axis=1)

text_cols = df.select_dtypes(include=["object", "string", "category"]).columns.drop("churn", errors="ignore")
print("Encoding:", list(text_cols))

df = pd.get_dummies(df, columns=text_cols, drop_first=True)

Encoding: ['contract_type', 'internet_service', 'payment_method']


In [6]:
# --------------------------------------------------
# 5. FEATURES AND TARGET
# --------------------------------------------------

X = df.drop("churn", axis=1)
y = df["churn"]

print(X.dtypes.value_counts())

bool       6
float64    3
int64      2
Name: count, dtype: int64


In [7]:
# --------------------------------------------------
# 6. TRAIN TEST SPLIT
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [8]:
# --------------------------------------------------
# 7a. LOGISTIC REGRESSION
# --------------------------------------------------

lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

In [9]:
# --------------------------------------------------
# 7b. DECISION TREE
# --------------------------------------------------

tree_model = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

In [10]:
# --------------------------------------------------
# 7c. RANDOM FOREST
# --------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

In [11]:
# --------------------------------------------------
# 8. EVALUATION FUNCTION
# --------------------------------------------------

def evaluate_model(model_name, actual, predicted):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, pos_label="Yes"),
        "Recall": recall_score(actual, predicted, pos_label="Yes"),
        "F1": f1_score(actual, predicted, pos_label="Yes")
    }

In [12]:
# --------------------------------------------------
# 9. COMPARE MODELS
# --------------------------------------------------

results = []

results.append(
    evaluate_model("Logistic Regression", y_test, lr_predictions)
)

results.append(
    evaluate_model("Decision Tree", y_test, tree_predictions)
)

results.append(
    evaluate_model("Random Forest", y_test, rf_predictions)
)

comparison_df = pd.DataFrame(results).round(3)

comparison_df.sort_values("Recall", ascending=False)

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.66,0.440,0.786,0.564
1,Decision Tree,0.63,0.415,0.786,0.543
2,Random Forest,0.69,0.465,0.714,0.563


In [13]:
# --------------------------------------------------
# 10. CONFUSION MATRICES
# --------------------------------------------------

for name, preds in [
    ("Logistic Regression", lr_predictions),
    ("Decision Tree", tree_predictions),
    ("Random Forest", rf_predictions)
]:
    print(name)
    print(confusion_matrix(y_test, preds, labels=["No", "Yes"]))
    print()

Logistic Regression
[[44 28]
 [ 6 22]]

Decision Tree
[[41 31]
 [ 6 22]]

Random Forest
[[49 23]
 [ 8 20]]



In [14]:
# --------------------------------------------------
# 11. OVERFITTING CHECK (train vs test accuracy)
# --------------------------------------------------

for name, m in [
    ("Logistic Regression", lr_model),
    ("Decision Tree", tree_model),
    ("Random Forest", rf_model)
]:
    print(f"{name:<20} train={m.score(X_train, y_train):.2f}  test={m.score(X_test, y_test):.2f}")

Logistic Regression  train=0.65  test=0.66
Decision Tree        train=0.71  test=0.63
Random Forest        train=0.84  test=0.69
